# Model: LightGBM

Owner: **Arman**

Parameter tuning on the existing features and data splits. Compare settings on 2018 validation data before evaluating the selected model on 2019.


In [8]:
import numpy as np
import pandas as pd
from pathlib import Path
from time import perf_counter
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

from lightgbm import LGBMRegressor


In [9]:
INCLUDE_RADIATION_DAY_BEFORE = True

MODEL_NAME = (
    "lightgbm_tuned"
    if INCLUDE_RADIATION_DAY_BEFORE
    else "lightgbm_no_radiation_day_before_tuned"
)


In [10]:
# using the shared train/validation/test sets

DATA_DIR = Path("../../data/NSW")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "nsw_train.csv", parse_dates=["DATETIME"])
validation = pd.read_csv(DATA_DIR / "nsw_validation.csv", parse_dates=["DATETIME"])
test = pd.read_csv(DATA_DIR / "nsw_test.csv", parse_dates=["DATETIME"])

print(train.shape, validation.shape, test.shape)

(140208, 55) (17520, 55) (17520, 55)


In [11]:
# TEMPERATURE/radiation are same-day actuals so not usable, only the day_before lags are

TARGET = "TOTALDEMAND"
DROP_COLS = ["DATETIME", TARGET, "TEMPERATURE", "radiation", "forecast_closest", "forecast_12hr_prior", "forecast_dayprior"]


if not INCLUDE_RADIATION_DAY_BEFORE:
    DROP_COLS.append("radiation_day_before")

FEATURES = [c for c in train.columns if c not in DROP_COLS]

len(FEATURES)

48

In [12]:
# Refit the original selected settings as a reference for this run.
model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42,
    verbosity=-1,
)

model.fit(train[FEATURES], train[TARGET])

print("Training finished")

Training finished


In [13]:
validation_predictions = model.predict(validation[FEATURES])

reference_rmse = np.sqrt(
    mean_squared_error(validation[TARGET], validation_predictions)
)

print(f"Reference validation RMSE: {reference_rmse:.2f} MW")


Reference validation RMSE: 418.75 MW


## Compare 12 parameter combinations

Keep the features, data periods and random seed fixed. Test 300 or 600 trees, 15 or 31 leaves, and learning rates of 0.03, 0.05 or 0.1. Select the lowest validation RMSE. Training times are approximate and depend on this computer.


In [14]:
results = []
best_rmse = float("inf")
best_model = None
best_params = None

for trees in [300, 600]:
    for leaves in [15, 31]:
        for rate in [0.03, 0.05, 0.1]:
            candidate = LGBMRegressor(
                n_estimators=trees,
                num_leaves=leaves,
                learning_rate=rate,
                random_state=42,
                verbosity=-1,
            )

            start = perf_counter()
            candidate.fit(train[FEATURES], train[TARGET])
            fit_seconds = perf_counter() - start
            val_predictions = candidate.predict(validation[FEATURES])

            rmse = np.sqrt(
                mean_squared_error(validation[TARGET], val_predictions)
            )

            results.append({
                "trees": trees,
                "leaves": leaves,
                "learning_rate": rate,
                "validation_rmse": rmse,
                "fit_seconds": fit_seconds,
            })
            print(f"{trees} trees, {leaves} leaves, rate {rate}: "
                  f"RMSE {rmse:.2f} MW, fit {fit_seconds:.1f}s")

            if rmse < best_rmse:
                best_rmse = rmse
                best_model = candidate
                best_params = {
                    "n_estimators": trees,
                    "num_leaves": leaves,
                    "learning_rate": rate,
                }

model = best_model
tuning_results = pd.DataFrame(results).sort_values("validation_rmse").reset_index(drop=True)
display(tuning_results)
print("Selected settings:", best_params)
print(f"Reference RMSE: {reference_rmse:.2f} MW; selected RMSE: {best_rmse:.2f} MW")

# Save the validation search for the report, separately from test scores.
tuning_results.to_csv(RESULTS_DIR / f"{MODEL_NAME}_validation_search.csv", index=False)


300 trees, 15 leaves, rate 0.03: RMSE 439.42 MW, fit 0.6s
300 trees, 15 leaves, rate 0.05: RMSE 431.53 MW, fit 0.6s
300 trees, 15 leaves, rate 0.1: RMSE 420.28 MW, fit 0.6s
300 trees, 31 leaves, rate 0.03: RMSE 426.83 MW, fit 0.9s
300 trees, 31 leaves, rate 0.05: RMSE 423.97 MW, fit 0.9s
300 trees, 31 leaves, rate 0.1: RMSE 418.75 MW, fit 0.8s
600 trees, 15 leaves, rate 0.03: RMSE 431.71 MW, fit 1.0s
600 trees, 15 leaves, rate 0.05: RMSE 424.77 MW, fit 1.1s
600 trees, 15 leaves, rate 0.1: RMSE 416.19 MW, fit 0.9s
600 trees, 31 leaves, rate 0.03: RMSE 422.59 MW, fit 1.6s
600 trees, 31 leaves, rate 0.05: RMSE 421.16 MW, fit 1.5s
600 trees, 31 leaves, rate 0.1: RMSE 422.19 MW, fit 1.4s


,trees,leaves,learning_rate,validation_rmse,fit_seconds
0,600,15,0.10,416.193537,0.915895
1,300,31,0.10,418.745182,0.792380
2,300,15,0.10,420.280441,0.562437
3,600,31,0.05,421.155866,1.478573
4,600,31,0.10,422.194155,1.442190
5,600,31,0.03,422.589548,1.641431
6,300,31,0.05,423.969270,0.937549
7,600,15,0.05,424.765226,1.093030
8,300,31,0.03,426.827084,0.908424
9,300,15,0.05,431.528451,0.572837


Selected settings: {'n_estimators': 600, 'num_leaves': 15, 'learning_rate': 0.1}
Reference RMSE: 418.75 MW; selected RMSE: 416.19 MW


## Focused follow-up: six combinations

The first round selected 600 trees, 15 leaves and a learning rate of 0.1. Keep the leaf count and learning rate fixed; compare 600 or 900 trees with min_child_samples of 20, 50 or 100. This parameter controls the minimum number of training examples in a leaf, approximately, with larger values discouraging rules based on small groups.

Run the first search before this cell. Retain the lowest validation RMSE across both rounds. The first round used the default min_child_samples of 20, so the 600-tree, 20-sample combination repeats its winner. This is the final planned search before test evaluation.


In [15]:
# Keep the winner from the first round unless validation improves.
followup_results = []

for trees in [600, 900]:
    for minimum_samples in [20, 50, 100]:
        candidate = LGBMRegressor(
            n_estimators=trees,
            num_leaves=15,
            learning_rate=0.1,
            min_child_samples=minimum_samples,
            random_state=42,
            verbosity=-1,
        )

        start = perf_counter()
        candidate.fit(train[FEATURES], train[TARGET])
        fit_seconds = perf_counter() - start
        val_predictions = candidate.predict(validation[FEATURES])
        rmse = np.sqrt(
            mean_squared_error(validation[TARGET], val_predictions)
        )

        followup_results.append({
            "trees": trees,
            "leaves": 15,
            "learning_rate": 0.1,
            "min_child_samples": minimum_samples,
            "validation_rmse": rmse,
            "fit_seconds": fit_seconds,
        })
        print(f"{trees} trees, minimum samples {minimum_samples}: "
              f"RMSE {rmse:.2f} MW, fit {fit_seconds:.1f}s")

        if rmse < best_rmse:
            best_rmse = rmse
            best_model = candidate

model = best_model
best_params = {
    name: model.get_params()[name]
    for name in ["n_estimators", "num_leaves", "learning_rate", "min_child_samples"]
}
followup_table = pd.DataFrame(followup_results).sort_values("validation_rmse").reset_index(drop=True)
display(followup_table)
print("Selected settings across both rounds:", best_params)
print(f"Reference RMSE: {reference_rmse:.2f} MW; selected RMSE: {best_rmse:.2f} MW")

followup_table.to_csv(RESULTS_DIR / f"{MODEL_NAME}_validation_followup.csv", index=False)


600 trees, minimum samples 20: RMSE 416.19 MW, fit 1.3s
600 trees, minimum samples 50: RMSE 423.62 MW, fit 1.1s
600 trees, minimum samples 100: RMSE 417.85 MW, fit 0.9s
900 trees, minimum samples 20: RMSE 418.54 MW, fit 1.2s
900 trees, minimum samples 50: RMSE 422.96 MW, fit 1.2s
900 trees, minimum samples 100: RMSE 420.38 MW, fit 1.3s


,trees,leaves,learning_rate,min_child_samples,validation_rmse,fit_seconds
0,600,15,0.1,20,416.193537,1.266082
1,600,15,0.1,100,417.845241,0.863101
2,900,15,0.1,20,418.540088,1.233136
3,900,15,0.1,100,420.379182,1.283419
4,900,15,0.1,50,422.962125,1.235085
5,600,15,0.1,50,423.624194,1.065494


Selected settings across both rounds: {'n_estimators': 600, 'num_leaves': 15, 'learning_rate': 0.1, 'min_child_samples': 20}
Reference RMSE: 418.75 MW; selected RMSE: 416.19 MW


## Review validation results before proceeding

Stop here to review the comparison. Run the remaining test and saving cells after settling the settings. Tuned predictions use a separate name so the original predictions remain available.


In [16]:
predictions = model.predict(test[FEATURES])
print("Number of predictions:", len(predictions))

Number of predictions: 17520


In [17]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2 = r2_score(y_true, y_pred)
    metrics = {"model": model_name, "rmse": rmse, "mae": mae, "mape_pct": mape, "r2": r2}
    print(metrics)
    return metrics


metrics = evaluate(test[TARGET], predictions, MODEL_NAME)

{'model': 'lightgbm_tuned', 'rmse': np.float64(471.758296954731), 'mae': 308.9318505243709, 'mape_pct': 3.7661566538752433, 'r2': 0.8575280131759251}


In [18]:
# saving prediction performance

pd.Series(predictions, index=test["DATETIME"], name=MODEL_NAME).to_csv(RESULTS_DIR / f"{MODEL_NAME}_predictions.csv")

comparison_path = RESULTS_DIR / "model_comparison.csv"
this_run = pd.DataFrame([metrics])

if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    existing = existing[existing["model"] != MODEL_NAME]
    this_run = pd.concat([existing, this_run], ignore_index=True)

this_run.to_csv(comparison_path, index=False)
this_run

,model,rmse,mae,mape_pct,r2,comments
0,xgboost,470.544565,311.176400,3.791660,0.858260,NaN
1,lightgbm,463.995139,307.403004,3.744587,0.862178,NaN
2,baseline_linear_regression,535.989120,380.215288,4.743792,0.816091,\r\nLinear Regression on the 48 shared feature...
3,prophet,469.383322,345.777006,4.289860,0.843310,NaN
4,random_forest_no_radiation_day_before,502.050043,326.322267,3.962717,0.838644,NaN
5,random_forest,493.696778,322.451412,3.927903,0.843969,NaN
6,aemo_forecast_closest,64.292411,47.711611,0.595688,0.997354,NaN
7,aemo_forecast_12hr_prior,212.913694,155.945260,1.921867,0.970980,NaN
8,aemo_forecast_dayprior,222.460754,161.830989,1.988398,0.968319,NaN
9,lightgbm_no_radiation_day_before,472.457778,311.011912,3.780578,0.857105,NaN


## LightGBM parameter tuning summary

Both the original and tuned models used the same 48 features, including
previous-day demand, temperature and radiation, plus calendar features.
Training used 2010–2017 data, validation used 2018, and testing used 2019.
Only model parameters changed; the features, data splits and random seed
remained fixed.

The first search compared 12 combinations of tree count, leaf count and
learning rate. A follow-up compared six combinations of tree count and
minimum samples per leaf. One configuration was repeated, giving
17 distinct configurations overall.

The settings selected by the lowest validation RMSE were:

- `n_estimators=600`
- `num_leaves=15`
- `learning_rate=0.1`
- `min_child_samples=20`

Both models were fitted on the training set only. The test set was used
for evaluation after selecting the parameters.

| Metric | Original | Tuned |
|---|---:|---:|
| Validation RMSE (MW) | 418.75 | 416.19 |
| Test RMSE (MW) | 464.00 | 471.76 |
| Test MAE (MW) | 307.40 | 308.93 |
| Test MAPE (%) | 3.7446 | 3.7662 |
| Test R² | 0.8622 | 0.8575 |

Tuning reduced validation RMSE by approximately 0.6%, but test RMSE
increased by approximately 1.7%. The small validation improvement did
not carry over to 2019, where the original model performed better on
all four test metrics.
